<a href="https://colab.research.google.com/github/MauryaAG07/Learning-and-practice/blob/main/chapter_appendix-tools-for-deep-learning/jupyter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import cupy as cp
import numpy as np
array_cpu = np.random.randint(0, 255, size = (4000,4000))
array_cpu.nbytes/1e6

128.0

In [2]:
array_gpu = cp.asarray(array_cpu)

In [3]:
%%timeit
cp.asarray(array_cpu)

36.7 ms ± 2.66 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [4]:
type(array_gpu)

cupy.ndarray

In [5]:
from scipy import fft

In [6]:
%%timeit
fft.fft(array_cpu)

390 ms ± 35.5 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [7]:
from cupyx.scipy import fft as fft_gpu

In [8]:
%%timeit
fft_gpu.fftn(array_gpu)

22.2 ms ± 221 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)


In [9]:

fft_cpu = fft.fftn(array_cpu)
fft_sent_back = cp.asnumpy(fft_gpu.fftn(array_gpu))
np.allclose(fft_sent_back, fft_cpu)

True

In [10]:
np.max(array_gpu)


array(254)

In [11]:
type(np.max(array_gpu))

cupy.ndarray

In [12]:
cp.random.randint(0,255, size=(4000,4000))

array([[ 83, 204, 252, ...,  38, 216,  50],
       [ 47, 112, 159, ..., 198,  94, 230],
       [ 64, 235, 161, ...,  19,  27, 219],
       ...,
       [163,  96, 213, ..., 121,  71, 203],
       [175, 108, 153, ..., 161,  85,  41],
       [176, 137, 202, ..., 116,  15,   8]])

It is much faster to generate arrays in the gpu vs cpu.

In [13]:
from numba import cuda

In [14]:
cuda.detect()

Found 1 CUDA devices
id 0                Tesla T4                              [SUPPORTED]
                      Compute Capability: 7.5
                           PCI Device ID: 4
                              PCI Bus ID: 0
                                    UUID: GPU-bbbf319f-51c6-0871-9111-72ecba3d8ef5
                                Watchdog: Disabled
             FP32/FP64 Performance Ratio: 32
Summary:
	1/1 devices are supported


True

In [15]:
cpu_array = np.random.randint(0,10, size=(2000,2000))

In [16]:
d_array = cuda.to_device(array_cpu)
d_array

In [17]:
cp.asarray(d_array)

array([[131,  55,  54, ...,  87, 217, 211],
       [ 96, 197, 160, ...,  25,  91,  77],
       [218, 118,   4, ..., 111, 251,  39],
       ...,
       [125, 215, 252, ...,  52, 248, 186],
       [219,  60, 151, ...,  46,  62,  31],
       [209,  16, 200, ...,  19,  67, 209]])

In [18]:
d_array.copy_to_host()

array([[131,  55,  54, ...,  87, 217, 211],
       [ 96, 197, 160, ...,  25,  91,  77],
       [218, 118,   4, ..., 111, 251,  39],
       ...,
       [125, 215, 252, ...,  52, 248, 186],
       [219,  60, 151, ...,  46,  62,  31],
       [209,  16, 200, ...,  19,  67, 209]])

In [19]:
from numba import cuda
import cupy as cp

In [20]:
@cuda.jit
def matmul(A, B, C):
  i, j = cuda.grid(2)
  if i < C.shape[0] and j < C.shape[1]:
    tmp = 0.
    for k in range(A.shape[1]):
      tmp += A[i,k] * B[k,j]
    C[i,j] = tmp

In [21]:
cp.random.seed(42)
A = cp.random.uniform(1,10,size=(2000,2000), dtype=np.float64)
B = cp.random.uniform(1,10,size=(2000,2000), dtype=np.float64)
C = cp.zeros((2000,2000), dtype=np.float64)

In [22]:
C

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [23]:
threadsperblock = (16,16)
blockspergrid_x = int(np.ceil(C.shape[0]/threadsperblock[0]))
blockspergrid_y = int(np.ceil(C.shape[0]/threadsperblock[1]))
blockspergrid = (blockspergrid_x, blockspergrid_y)
print(blockspergrid)
print(f"This is the number of threads that we will be executing: {threadsperblock[0]*blockspergrid_x}")

(125, 125)
This is the number of threads that we will be executing: 2000


In [24]:
matmul[blockspergrid, threadsperblock](A, B, C)

In [25]:
C

array([[59394.46607842, 58001.66377549, 58910.89964126, ...,
        58755.23643036, 59265.65525416, 58447.86197932],
       [59656.82462269, 58635.04995946, 59080.54393462, ...,
        59327.90030958, 60391.24930458, 59425.35827899],
       [62192.77335924, 60700.17680915, 60538.34933653, ...,
        61027.03460329, 61711.10155029, 60544.69882075],
       ...,
       [60649.27416407, 59951.20972379, 60170.2004206 , ...,
        60203.88074659, 60934.19598791, 59613.28418599],
       [61620.11922557, 61264.33868343, 62076.33462258, ...,
        61227.57661876, 62642.97523374, 61841.46799761],
       [61535.95697543, 59600.43760873, 59927.620961  , ...,
        60738.55627077, 61429.70009593, 59662.34901713]])

This is considered the slower way which doesnt take full advantage of shared memory. Next we will begin implementing a custom kernel which uses tiling, meant to acheive more optimal speed.

In [26]:
from numba import float32, int32, float64

TPB = 16


@cuda.jit
def fast_matmul(A, B, C):
  sA = cuda.shared.array(shape=(TPB,TPB),dtype=float32)
  sB = cuda.shared.array(shape=(TPB,TPB),dtype=float32)
  x, y = cuda.grid(2)

  tx = cuda.threadIdx.x
  ty = cuda.threadIdx.y
  bpg = cuda.gridDim.x

  if x>= C.shape[0] and y >= C.shape[1]:
    return
  tmp = 0.

  for k in range(bpg):
    sA[tx,ty] = A[x, ty + k*TPB]
    sB[tx,ty] = B[tx + k* TPB, y]
    cuda.syncthreads()

    for j in range(TPB):
      tmp += sA[tx,j]* sB[j,ty]
      cuda.syncthreads()
    C[x,y] = tmp


In [27]:
SIZE = 4000
A = cp.random.uniform(size =(SIZE,SIZE),dtype = np.float32)
B = cp.random.uniform(size =(SIZE,SIZE),dtype = np.float32)
C_slow = cp.zeros((SIZE, SIZE), dtype=np.float32)
C_fast = cp.zeros((SIZE, SIZE), dtype=np.float32)

In [28]:
threadsperblock = (TPB,TPB)
blockspergrid = int(np.ceil(SIZE/threadsperblock[0]))
blockspergrid = (blockspergrid, blockspergrid)

In [29]:
matmul[blockspergrid, threadsperblock](A, B, C_slow)

In [30]:
fast_matmul[blockspergrid, threadsperblock](A, B, C_fast)

In [31]:
cp.allclose(C_slow, C_fast)

array(True)

In [36]:
%%time
for i in range(10):
  matmul[blockspergrid, threadsperblock](A, B, C_slow)

CPU times: user 13.6 s, sys: 7.55 ms, total: 13.6 s
Wall time: 13.6 s


In [37]:
%%time
for i in range(10):
  fast_matmul[blockspergrid, threadsperblock](A, B, C_fast)

CPU times: user 12.4 s, sys: 296 µs, total: 12.4 s
Wall time: 12.4 s


In [34]:
C_c = cp.dot(A, B)
cp.allclose(C_c, C_fast)

array(True)

In [38]:
%%time
for i in range(10):
  cp.dot(A,B)

CPU times: user 2.02 ms, sys: 0 ns, total: 2.02 ms
Wall time: 1.6 ms
